# Apache Spark RDDs — Hands-on Notebook
**Analytics Foundation · Data Engineering Track**

This notebook runs top-to-bottom on **Google Colab** (Runtime → Run all). It covers:

1. Creating RDDs
2. RDD Operations — Transformations and Actions
3. Lazy Evaluation and the DAG
4. Key-Value Pair RDDs
5. Aggregations and Reductions
6. Joins and Co-Grouping
7. Caching and Persistence

> Tip: read the markdown above each cell, *predict* the output, then run it.

## 0. Setup
Installs PySpark 3.5.3 (pinned so results are reproducible) and makes sure a compatible Java (8/11/17) is available.

In [ ]:
# --- Setup: Java + PySpark (safe to re-run) ---
import os, re, shutil, subprocess, sys

def java_major():
    """Return the installed Java major version, or None if Java is missing."""
    if shutil.which("java") is None:
        return None
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
    if not m:
        return None
    major = int(m.group(1))
    return int(m.group(2)) if major == 1 else major   # "1.8" -> 8

jv = java_major()
if jv not in (8, 11, 17):
    print(f"Java version {jv} not suitable for Spark 3.5 -> installing OpenJDK 17 ...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
else:
    print(f"Java {jv} found - OK")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)
print("PySpark installed")

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("RDD-Fundamentals")
         .master("local[*]")          # use all cores of the Colab machine
         .getOrCreate())

sc = spark.sparkContext               # RDDs live on the SparkContext
sc.setLogLevel("ERROR")               # hide noisy INFO/WARN logs

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Default cores :", sc.defaultParallelism)

---
## 1. Creating RDDs
An **RDD** is an immutable collection of records split into **partitions** spread across the cluster.
There are three common ways to create one:

| Method | Use it when |
|---|---|
| `sc.parallelize(list)` | testing / small data already in Python |
| `sc.textFile(path)` | reading files (each line = one record) |
| `df.rdd` | converting an existing DataFrame |

### 1.1 From a Python collection — `parallelize`

In [ ]:
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], numSlices=3)

print("Type        :", type(numbers))
print("Partitions  :", numbers.getNumPartitions())
print("Data        :", numbers.collect())
print("Per partition (glom):", numbers.glom().collect())

### 1.2 From a text file — `textFile`
We first create a small sample file so the notebook is self-contained.

In [ ]:
sample_text = """data engineering is fun
big data requires distributed processing
hadoop mapreduce processes big data
spark is faster than mapreduce
data engineering uses spark and hadoop"""

with open("sample.txt", "w") as f:
    f.write(sample_text)

lines = sc.textFile("sample.txt")
print("Number of lines:", lines.count())
for line in lines.collect():
    print(" ->", line)

### 1.3 From a DataFrame — `df.rdd`
Every DataFrame has an RDD of `Row` objects underneath.

In [ ]:
df = spark.createDataFrame([("Alice", 25), ("Bob", 30), ("Cathy", 29)], ["name", "age"])
df.show()

people_rdd = df.rdd                      # RDD of Row objects
print(people_rdd.collect())
print("Names only:", people_rdd.map(lambda row: row["name"]).collect())

### 1.4 Other handy creators

In [ ]:
empty = sc.emptyRDD()
print("Empty RDD count:", empty.count())

r = sc.range(0, 10, 2)                   # like Python range(0, 10, 2)
print("sc.range       :", r.collect())

---
## 2. RDD Operations — Transformations and Actions
* **Transformation** → returns a *new RDD*, runs **lazily** (just a recipe). e.g. `map`, `filter`, `flatMap`, `distinct`, `union`
* **Action** → returns a *result* to the driver (or writes output) and **triggers execution**. e.g. `collect`, `count`, `take`, `first`, `reduce`

### 2.1 Common transformations

In [ ]:
nums = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

squares = nums.map(lambda x: x * x)            # one output per input
evens   = nums.filter(lambda x: x % 2 == 0)    # keep only matching
print("map    (squares):", squares.collect())
print("filter (evens)  :", evens.collect())

### 2.2 `map` vs `flatMap` — the classic interview question

In [ ]:
sentences = sc.parallelize(["spark is fast", "rdd is resilient"])

print("map     ->", sentences.map(lambda s: s.split(" ")).collect())      # list of lists
print("flatMap ->", sentences.flatMap(lambda s: s.split(" ")).collect())  # flattened

### 2.3 Set-style transformations

In [ ]:
a = sc.parallelize([1, 2, 3, 4, 4, 5])
b = sc.parallelize([4, 5, 6, 7])

print("distinct     :", sorted(a.distinct().collect()))
print("union        :", a.union(b).collect())              # keeps duplicates
print("intersection :", sorted(a.intersection(b).collect()))
print("subtract     :", sorted(a.subtract(b).collect()))
print("sortBy desc  :", a.sortBy(lambda x: x, ascending=False).collect())

### 2.4 Common actions

In [ ]:
nums = sc.parallelize([5, 3, 8, 1, 9, 2])

print("collect      :", nums.collect())
print("count        :", nums.count())
print("first        :", nums.first())
print("take(3)      :", nums.take(3))
print("top(2)       :", nums.top(2))
print("takeOrdered  :", nums.takeOrdered(3))
print("reduce (sum) :", nums.reduce(lambda a, b: a + b))
print("countByValue :", dict(sc.parallelize(["a", "b", "a", "c", "a"]).countByValue()))

> ⚠️ **`collect()` brings ALL data to the driver.** Fine for small demos, dangerous on big data.
> Prefer `take(n)` to peek.

---
## 3. Lazy Evaluation and the DAG
Transformations are **not executed** when you write them. Spark only records them in a **DAG
(Directed Acyclic Graph)** — the lineage of how each RDD is built. When an **action** is called,
Spark optimises the whole DAG, splits it into **stages** at every **shuffle**, and runs it.

### 3.1 Proof that transformations are lazy
The bad transformation below divides by zero — but no error appears until an action runs!

In [ ]:
import time

data = sc.parallelize(range(1, 6))

start = time.time()
broken = data.map(lambda x: x / 0)          # transformation: nothing runs yet
print(f"Transformation defined in {time.time() - start:.4f}s - no error so far!")

try:
    broken.collect()                        # action: NOW Spark executes
except Exception as e:
    print("Error appears only at the action:", type(e).__name__)

### 3.2 Seeing the DAG (lineage) with `toDebugString`

In [ ]:
words = (sc.textFile("sample.txt")
           .flatMap(lambda line: line.split(" "))   # narrow
           .map(lambda w: (w, 1)))                  # narrow
counts = words.reduceByKey(lambda a, b: a + b)      # WIDE -> shuffle -> new stage

print("Lineage of `words` (narrow only):")
print(words.toDebugString().decode("utf-8"))
print("\nLineage of `counts` (notice the extra indented block = shuffle/stage boundary):")
print(counts.toDebugString().decode("utf-8"))

**Narrow vs Wide transformations**

| Narrow (no shuffle) | Wide (shuffle → new stage) |
|---|---|
| `map`, `filter`, `flatMap`, `mapValues`, `union` | `reduceByKey`, `groupByKey`, `join`, `distinct`, `sortByKey`, `repartition` |

Narrow steps get *pipelined* together into one stage. Wide steps move data over the network — the most expensive part of any Spark job.

In [ ]:
# The action finally triggers the 2-stage job described above
print(sorted(counts.collect(), key=lambda kv: -kv[1])[:5])

---
## 4. Key-Value Pair RDDs
When every element is a 2-tuple `(key, value)`, Spark unlocks the **"ByKey"** family of operations.
Real-world examples: `(customer_id, amount)`, `(city, temperature)`, `(word, 1)`.

In [ ]:
sales = sc.parallelize([
    ("Electronics", 1200), ("Clothing", 300), ("Electronics", 800),
    ("Grocery", 150), ("Clothing", 450), ("Grocery", 200), ("Electronics", 500),
])

print("keys     :", sales.keys().collect())
print("values   :", sales.values().collect())
print("mapValues (add 18% GST):", sales.mapValues(lambda v: round(v * 1.18, 2)).collect())
print("sortByKey:", sales.sortByKey().collect())
print("countByKey:", dict(sales.countByKey()))
print("lookup('Grocery'):", sales.lookup("Grocery"))

### 4.1 Building a pair RDD from raw text

In [ ]:
raw = sc.parallelize(["101,Mumbai,250", "102,Delhi,400", "103,Mumbai,150", "104,Pune,300"])

city_amount = raw.map(lambda line: line.split(",")) \
                 .map(lambda f: (f[1], int(f[2])))      # (city, amount)
print(city_amount.collect())

### 4.2 `flatMapValues` — one key, many values

In [ ]:
skills = sc.parallelize([("Asha", "python,sql"), ("Ravi", "spark,scala,sql")])
print(skills.flatMapValues(lambda s: s.split(",")).collect())

---
## 5. Aggregations and Reductions
### 5.1 Whole-RDD reductions (actions)
`reduce`, `fold`, `sum`, `max`, `min`, `mean`, `stats`. The function passed to `reduce` must be
**associative and commutative**, because Spark combines partitions in no fixed order.

In [ ]:
nums = sc.parallelize([10, 20, 30, 40, 50], 2)

print("reduce :", nums.reduce(lambda a, b: a + b))
print("fold   :", nums.fold(0, lambda a, b: a + b))
print("sum    :", nums.sum())
print("max/min:", nums.max(), nums.min())
print("mean   :", nums.mean())
print("stats  :", nums.stats())

### 5.2 Common mistake: `reduce` on a pair RDD
`reduce` treats each `(word, 1)` tuple as one value, so `a + b` **concatenates tuples** — not what we want.
It is also an *action*, so its result is a Python tuple, not an RDD.

In [ ]:
pairs = sc.parallelize([("data", 1), ("spark", 1), ("data", 1)])
wrong = pairs.reduce(lambda a, b: a + b)
print("reduce on pairs  ->", wrong, "| type:", type(wrong).__name__)

right = pairs.reduceByKey(lambda a, b: a + b)
print("reduceByKey      ->", sorted(right.collect()))

### 5.3 Per-key aggregations: `reduceByKey`, `groupByKey`, `foldByKey`

In [ ]:
total_by_cat = sales.reduceByKey(lambda a, b: a + b)
print("reduceByKey (total):", sorted(total_by_cat.collect()))

grouped = sales.groupByKey().mapValues(list)
print("groupByKey (lists) :", sorted(grouped.collect()))

print("foldByKey (total)  :", sorted(sales.foldByKey(0, lambda a, b: a + b).collect()))
print("max per category   :", sorted(sales.reduceByKey(max).collect()))

> 💡 **Prefer `reduceByKey` over `groupByKey`.** `reduceByKey` pre-aggregates inside each partition
> *before* the shuffle (map-side combine), so far less data crosses the network.

### 5.4 Average per key — `aggregateByKey` and `combineByKey`
The output type `(sum, count)` differs from the input type (a number), so we need these more flexible tools.

In [ ]:
# aggregateByKey(zeroValue, seqFunc, combFunc)
sum_count = sales.aggregateByKey(
    (0, 0),                                          # zero value: (sum, count)
    lambda acc, v: (acc[0] + v, acc[1] + 1),         # within a partition
    lambda a, b: (a[0] + b[0], a[1] + b[1]))         # across partitions
avg = sum_count.mapValues(lambda sc_: round(sc_[0] / sc_[1], 2))
print("aggregateByKey avg:", sorted(avg.collect()))

# combineByKey(createCombiner, mergeValue, mergeCombiners)
avg2 = sales.combineByKey(
    lambda v: (v, 1),
    lambda acc, v: (acc[0] + v, acc[1] + 1),
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(lambda t: round(t[0] / t[1], 2))
print("combineByKey avg  :", sorted(avg2.collect()))

### 5.5 Capstone: Word Count (the 'Hello World' of big data)

In [ ]:
word_counts = (sc.textFile("sample.txt")
                 .flatMap(lambda line: line.lower().split())
                 .map(lambda w: (w, 1))
                 .reduceByKey(lambda a, b: a + b)
                 .sortBy(lambda kv: (-kv[1], kv[0])))

for word, cnt in word_counts.take(8):
    print(f"{word:<12} {cnt}")

---
## 6. Joins and Co-Grouping
Joins work on **pair RDDs** and match records by **key**. Every join is a wide transformation (shuffle).

In [ ]:
customers = sc.parallelize([(1, "Asha"), (2, "Ravi"), (3, "Meena"), (4, "John")])
orders    = sc.parallelize([(1, 500), (1, 300), (2, 700), (5, 900)])   # customer 5 does not exist

print("inner join      :", sorted(customers.join(orders).collect()))
print("leftOuterJoin   :", sorted(customers.leftOuterJoin(orders).collect()))
print("rightOuterJoin  :", sorted(customers.rightOuterJoin(orders).collect()))
print("fullOuterJoin   :", sorted(customers.fullOuterJoin(orders).collect()))
print("subtractByKey   :", sorted(customers.subtractByKey(orders).collect()), "<- customers with no orders")

### 6.1 `cogroup` — group two RDDs by key in one pass
Returns `(key, (iterable_from_rdd1, iterable_from_rdd2))`. Useful when you need **all** values from both sides,
e.g. a customer's name *and* every order.

In [ ]:
cg = customers.cogroup(orders).mapValues(lambda pair: (list(pair[0]), list(pair[1])))
for key, (names, amts) in sorted(cg.collect()):
    print(f"customer {key}: names={names}, orders={amts}, total={sum(amts)}")

### 6.2 Broadcast join (map-side join) — avoid the shuffle
If one side is **small**, send a read-only copy to every executor with `sc.broadcast`, then use a plain `map`.

In [ ]:
city_lookup = sc.broadcast({"MUM": "Mumbai", "DEL": "Delhi", "BLR": "Bengaluru"})

trips = sc.parallelize([("T1", "MUM"), ("T2", "BLR"), ("T3", "DEL"), ("T4", "XYZ")])
enriched = trips.map(lambda t: (t[0], city_lookup.value.get(t[1], "Unknown")))
print(enriched.collect())

---
## 7. Caching and Persistence
Because of lazy evaluation, **every action recomputes the RDD from scratch** using its lineage.
If you reuse an RDD in several actions, store it with `cache()` / `persist()`.

| Storage level | Meaning |
|---|---|
| `MEMORY_ONLY` (default for `cache()` on RDDs) | keep in RAM; recompute partitions that don't fit |
| `MEMORY_AND_DISK` | spill to disk what doesn't fit in RAM |
| `DISK_ONLY` | store only on disk |
| `MEMORY_ONLY_2`, `MEMORY_AND_DISK_2` | same, replicated on 2 nodes |

In [ ]:
import time

def slow_square(x):
    s = 0
    for _ in range(50):          # simulate expensive work
        s += x * x
    return s // 50

big = sc.parallelize(range(300_000), 4).map(slow_square)

t = time.time(); big.count(); print(f"1st count (no cache)      : {time.time()-t:.2f}s")
t = time.time(); big.count(); print(f"2nd count (no cache)      : {time.time()-t:.2f}s  <- recomputed")

big.cache()                                   # mark for caching (lazy!)
t = time.time(); big.count(); print(f"3rd count (fills cache)   : {time.time()-t:.2f}s")
t = time.time(); big.count(); print(f"4th count (from cache)    : {time.time()-t:.2f}s  <- faster")

print("is_cached     :", big.is_cached)
print("storage level :", big.getStorageLevel())

In [ ]:
from pyspark import StorageLevel

big.unpersist()                                       # free the memory first
big.persist(StorageLevel.MEMORY_AND_DISK)             # choose a storage level explicitly
big.count()                                           # action materialises it
print("storage level now:", big.getStorageLevel())

big.unpersist()
print("after unpersist, is_cached =", big.is_cached)

**When to cache**
* ✅ An RDD used by **more than one action** (e.g. iterative ML, multiple reports from one cleaned dataset)
* ✅ After an expensive step (a join or a heavy parse)
* ❌ Data used only once — caching just wastes memory
* Always `unpersist()` when done.

---
## 8. Practice Exercises
Try these before looking at the solution cell below.
1. Create an RDD of numbers 1–20 and return the sum of squares of odd numbers.
2. From `sales`, find the category with the highest total revenue.
3. Join `customers` and `orders` and return each customer's **total** spend (customers with no orders = 0).
4. Count how many words in `sample.txt` have more than 5 letters.

In [ ]:
# ---------- Solutions ----------
# 1
print("1)", sc.parallelize(range(1, 21)).filter(lambda x: x % 2).map(lambda x: x * x).sum())

# 2
print("2)", sales.reduceByKey(lambda a, b: a + b).max(key=lambda kv: kv[1]))

# 3
spend = (customers.leftOuterJoin(orders)
                  .map(lambda kv: ((kv[0], kv[1][0]), kv[1][1] or 0))
                  .reduceByKey(lambda a, b: a + b))
print("3)", sorted(spend.collect()))

# 4
print("4)", sc.textFile("sample.txt").flatMap(lambda l: l.split()).filter(lambda w: len(w) > 5).count())

---
### Clean up
Always stop the session at the very end (and only at the end!).

In [ ]:
spark.stop()
print("Spark session stopped.")